## Week 5

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [52]:
df1_animation = pd.read_csv("data/Animation_Movies_clean.csv")

In [53]:
df1_animation["log_budget"] = np.where(
    df1_animation["budget"] > 0,
    np.log10(df1_animation["budget"]),
    np.nan
)

df1_animation["log_popularity"] = np.where(
    df1_animation["popularity"] > 0,
    np.log10(df1_animation["popularity"]),
    np.nan
)

df1_animation["log_revenue"] = np.where(
    df1_animation["revenue"] > 0,
    np.log10(df1_animation["revenue"]),
    np.nan
)

/usr/local/python/3.14.2/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [54]:
animation_week5 = df1_animation.dropna(
    subset=[
        "log_budget",
        "log_popularity",
        "vote_average",
        "vote_count",
        "runtime",
        "log_revenue"
    ]
).copy()

animation_week5.shape

(616, 23)

In [55]:
revenue_median = animation_week5[
    "log_revenue"
].median()

animation_week5["high_revenue"] = (
    animation_week5["log_revenue"] > revenue_median
).astype(int)

In [56]:
animation_week5["high_revenue"].value_counts()

high_revenue
1    308
0    308
Name: count, dtype: int64

In [57]:
features = [
    "log_budget",
    "log_popularity",
    "vote_average",
    "vote_count",
    "runtime"
]

X = animation_week5[features]

y = animation_week5["high_revenue"]

In [58]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [59]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [60]:
linear_svm = SVC(
    kernel="linear",
    C=1
)

linear_svm.fit(
    X_train_scaled,
    y_train
)

y_pred_linear = linear_svm.predict(
    X_test_scaled
)

In [61]:
linear_accuracy = accuracy_score(
    y_test,
    y_pred_linear
)

print("Linear SVM Accuracy:", linear_accuracy)

Linear SVM Accuracy: 0.8951612903225806


In [62]:
linear_cm = confusion_matrix(
    y_test,
    y_pred_linear
)

linear_cm

array([[54,  8],
       [ 5, 57]])

In [63]:
print(
    classification_report(
        y_test,
        y_pred_linear
    )
)

              precision    recall  f1-score   support

           0       0.92      0.87      0.89        62
           1       0.88      0.92      0.90        62

    accuracy                           0.90       124
   macro avg       0.90      0.90      0.90       124
weighted avg       0.90      0.90      0.90       124



In [64]:
rbf_svm = SVC(
    kernel="rbf",
    C=1
)

rbf_svm.fit(
    X_train_scaled,
    y_train
)

y_pred_rbf = rbf_svm.predict(
    X_test_scaled
)

In [65]:
rbf_accuracy = accuracy_score(
    y_test,
    y_pred_rbf
)

print("RBF SVM Accuracy:", rbf_accuracy)

RBF SVM Accuracy: 0.8951612903225806


In [66]:
print(
    confusion_matrix(
        y_test,
        y_pred_rbf
    )
)

[[54  8]
 [ 5 57]]


In [67]:
print(
    classification_report(
        y_test,
        y_pred_rbf
    )
)

              precision    recall  f1-score   support

           0       0.92      0.87      0.89        62
           1       0.88      0.92      0.90        62

    accuracy                           0.90       124
   macro avg       0.90      0.90      0.90       124
weighted avg       0.90      0.90      0.90       124



In [68]:
poly_svm = SVC(
    kernel="poly",
    degree=2,
    C=1
)

poly_svm.fit(
    X_train_scaled,
    y_train
)

y_pred_poly = poly_svm.predict(
    X_test_scaled
)

In [69]:
poly_accuracy = accuracy_score(
    y_test,
    y_pred_poly
)

print("Polynomial SVM Accuracy:", poly_accuracy)

Polynomial SVM Accuracy: 0.7096774193548387


In [70]:
kernel_results = pd.DataFrame({
    "Kernel": [
        "Linear",
        "RBF",
        "Polynomial"
    ],
    "Accuracy": [
        linear_accuracy,
        rbf_accuracy,
        poly_accuracy
    ]
})

kernel_results

,Kernel,Accuracy
0,Linear,0.895161
1,RBF,0.895161
2,Polynomial,0.709677


### SVM Regularization

In [71]:
c_values = [
    0.01,
    0.1,
    1,
    10,
    100
]

c_results = []

for c in c_values:

    model = SVC(
        kernel="rbf",
        C=c
    )

    model.fit(
        X_train_scaled,
        y_train
    )

    predictions = model.predict(
        X_test_scaled
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    c_results.append([
        c,
        accuracy
    ])

In [72]:
regularization_results = pd.DataFrame(
    c_results,
    columns=[
        "C",
        "Accuracy"
    ]
)

regularization_results

,C,Accuracy
0,0.01,0.879032
1,0.10,0.887097
2,1.00,0.895161
3,10.00,0.895161
4,100.00,0.895161


### Dataframe 2

In [73]:
df2_action = pd.read_csv("data/Action_Movies_clean.csv")

In [74]:
df2_action["log_votes"] = np.log10(
    df2_action["votes"]
)

df2_action["log_gross"] = np.log10(
    df2_action["gross(in $)"]
)

In [75]:
action_week5 = df2_action.dropna(
    subset=[
        "log_votes",
        "rating",
        "runtime",
        "year",
        "log_gross"
    ]
).copy()

action_week5.shape

(2758, 16)

In [76]:
gross_median = action_week5[
    "log_gross"
].median()

action_week5["high_gross"] = (
    action_week5["log_gross"] > gross_median
).astype(int)

In [77]:
action_week5["high_gross"].value_counts()

high_gross
1    1379
0    1379
Name: count, dtype: int64

In [78]:
features = [
    "log_votes",
    "rating",
    "runtime",
    "year"
]

X = action_week5[features]

y = action_week5["high_gross"]

In [79]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [80]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)

In [81]:
linear_svm = SVC(
    kernel="linear",
    C=1
)

linear_svm.fit(
    X_train_scaled,
    y_train
)

y_pred_linear = linear_svm.predict(
    X_test_scaled
)

In [82]:
linear_accuracy = accuracy_score(
    y_test,
    y_pred_linear
)

print(
    "Linear SVM Accuracy:",
    linear_accuracy
)

Linear SVM Accuracy: 0.8496376811594203


In [83]:
confusion_matrix(
    y_test,
    y_pred_linear
)

array([[227,  49],
       [ 34, 242]])

In [84]:
print(
    classification_report(
        y_test,
        y_pred_linear
    )
)

              precision    recall  f1-score   support

           0       0.87      0.82      0.85       276
           1       0.83      0.88      0.85       276

    accuracy                           0.85       552
   macro avg       0.85      0.85      0.85       552
weighted avg       0.85      0.85      0.85       552



In [85]:
rbf_svm = SVC(
    kernel="rbf",
    C=1
)

rbf_svm.fit(
    X_train_scaled,
    y_train
)

y_pred_rbf = rbf_svm.predict(
    X_test_scaled
)

rbf_accuracy = accuracy_score(
    y_test,
    y_pred_rbf
)

print(
    "RBF SVM Accuracy:",
    rbf_accuracy
)

RBF SVM Accuracy: 0.8822463768115942


In [86]:
poly_svm = SVC(
    kernel="poly",
    degree=2,
    C=1
)

poly_svm.fit(
    X_train_scaled,
    y_train
)

y_pred_poly = poly_svm.predict(
    X_test_scaled
)

poly_accuracy = accuracy_score(
    y_test,
    y_pred_poly
)

print(
    "Polynomial SVM Accuracy:",
    poly_accuracy
)

Polynomial SVM Accuracy: 0.6739130434782609


In [87]:
action_kernel_results = pd.DataFrame({
    "Kernel": [
        "Linear",
        "RBF",
        "Polynomial"
    ],
    "Accuracy": [
        linear_accuracy,
        rbf_accuracy,
        poly_accuracy
    ]
})

action_kernel_results

,Kernel,Accuracy
0,Linear,0.849638
1,RBF,0.882246
2,Polynomial,0.673913


In [88]:
c_values = [
    0.01,
    0.1,
    1,
    10,
    100
]

c_results = []

for c in c_values:

    model = SVC(
        kernel="rbf",
        C=c
    )

    model.fit(
        X_train_scaled,
        y_train
    )

    predictions = model.predict(
        X_test_scaled
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    c_results.append([
        c,
        accuracy
    ])

In [89]:
action_regularization_results = pd.DataFrame(
    c_results,
    columns=[
        "C",
        "Accuracy"
    ]
)

action_regularization_results

,C,Accuracy
0,0.01,0.829710
1,0.10,0.884058
2,1.00,0.882246
3,10.00,0.882246
4,100.00,0.875000
